In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "README.md").is_file() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "README.md").is_file():
    raise FileNotFoundError("Run this notebook from the project directory or one of its subdirectories.")


# EXP10 — PatchTST + Competition Aligned Weighted RMSE Loss

이 노트북은 물리 정제가 완료된 `train_final_physics_v3.csv`를 입력으로 사용하여 **PatchTST (Patch Time Series Transformer)** 모델을 학습합니다.

### EXP10 핵심 구성
1. **PatchTST 아키텍처 적용**: 289스텝(48시간) 시계열을 패치(Patch) 단위로 분할 임베딩하여 로컬 시계열 패턴과 장기 의존성을 동시에 학습
2. **Feature Ablation Study (RAM-Safe)**: 16개 물리 피처 세트 중 PatchTST에 가장 최적화된 조합 탐색
3. **CompetitionAlignedRMSELoss 연동**: $h_s \ge 1.5\text{m}$ 고파랑 가중치 손실함수 적용
4. **Optuna 하이퍼파라미터 튜닝** (`patch_len`, `stride`, `d_model`, `n_heads`, `e_layers`, `lr` 등)
5. **`test_context.parquet` 추론 및 `submission_exp10.csv` 생성**

In [3]:
# ============================================================
# 0. SETUP & PACKAGES
# ============================================================
!pip -q install optuna

from pathlib import Path
import gc
import json
import math
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import optuna
from optuna.samplers import TPESampler

warnings.filterwarnings("ignore")

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 28.4 MB/s eta 0:00:00
DEVICE: cuda
GPU: Tesla T4


In [4]:
# ============================================================
# 1. CONFIG
# ============================================================
DATA_PATH = Path(PROJECT_ROOT / "data" / "processed" / "train_final_physics_v3.csv")
TEST_CONTEXT_PATH = Path(PROJECT_ROOT / "data" / "raw" / "test_context.parquet")
TEST_INDEX_PATH = Path(PROJECT_ROOT / "data" / "raw" / "test_index.csv")
SUBMISSION_PATH = Path(PROJECT_ROOT / "submission" / "submission_exp10.csv")

EXP_DIR = PROJECT_ROOT / "artifacts" / "experiments" / "exp10"
EXP_DIR.mkdir(parents=True, exist_ok=True)

TIME_COL = "time"
STATION_COL = "station"

STEP_MINUTES = 10
STEPS_PER_HOUR = 6

INPUT_LEN = 289
LEAD_HOURS = [3, 6, 9, 12, 18, 24]
LEAD_STEPS = [h * STEPS_PER_HOUR for h in LEAD_HOURS]
MAX_LEAD = max(LEAD_STEPS)
N_TARGETS = len(LEAD_STEPS)

TRAIN_RATIO = 0.80

ABLATION_MAX_SAMPLES = 25000
ABLATION_EPOCHS = 12
ABLATION_PATIENCE = 3

N_TRIALS = 30
OPTUNA_EPOCHS = 18
OPTUNA_PATIENCE = 4

FINAL_EPOCHS = 35
FINAL_PATIENCE = 6

NUM_WORKERS = 0

print("INPUT_LEN:", INPUT_LEN)
print("LEADS:", LEAD_HOURS)

INPUT_LEN: 289
LEADS: [3, 6, 9, 12, 18, 24]


In [5]:
# ============================================================
# 2. LOAD DATA
# ============================================================
df = pd.read_csv(DATA_PATH, parse_dates=[TIME_COL])
df = df.sort_values([STATION_COL, TIME_COL]).reset_index(drop=True)

if "hs_original_observed" not in df.columns:
    df["hs_original_observed"] = df["hs"].notna().astype(np.int8)

print("shape:", df.shape)
print("stations:", df[STATION_COL].unique())
print("period:", df[TIME_COL].min(), "~", df[TIME_COL].max())

shape: (183600, 36)
stations: ['G-ORS' 'I-ORS' 'S-ORS']
period: 2024-01-01 00:00:00+09:00 ~ 2025-06-30 23:50:00+09:00


In [6]:
# ============================================================
# 3. EXPANDED FEATURE GROUPS & CANDIDATE SETS
# ============================================================
FEATURE_GROUPS = {
    "BASE_WAVE": ["hs", "tp", "hmax"],
    "WIND_BASIC": ["wspd", "gust", "u_wind", "v_wind"],
    "WAVE_MOMENTUM": ["hs_diff_1h", "hs_diff_3h", "hs_mean_6h", "hs_mean_12h", "hs_max_6h", "hs_max_12h"],
    "WAVE_DYNAMICS": ["wave_steepness", "wave_energy", "effective_wind_forcing", "u_wave", "v_wave"],
    "WIND_HISTORY": ["wspd_mean_6h", "wspd_mean_12h", "gust_max_6h", "gust_max_12h", "gust_minus_wspd"],
    "PRESSURE_TENDENCY": ["caph_change_3h", "caph_change_6h", "caph_change_12h"],
    "ALIGNMENT": ["wind_wave_alignment", "wind_wave_diff"],
    "ATMOS": ["airt", "relh", "caph"],
}

for k, cols in list(FEATURE_GROUPS.items()):
    FEATURE_GROUPS[k] = [c for c in cols if c in df.columns]

def uniq(seq):
    return list(dict.fromkeys(seq))

FEATURE_SETS = {}
FEATURE_SETS["BASE_WAVE"] = uniq(FEATURE_GROUPS["BASE_WAVE"])
FEATURE_SETS["BASE_WAVE_WIND"] = uniq(FEATURE_GROUPS["BASE_WAVE"] + FEATURE_GROUPS["WIND_BASIC"])
FEATURE_SETS["BASE_WAVE_MOMENTUM"] = uniq(FEATURE_GROUPS["BASE_WAVE"] + FEATURE_GROUPS["WAVE_MOMENTUM"])
FEATURE_SETS["BASE_WAVE_DYNAMICS"] = uniq(FEATURE_GROUPS["BASE_WAVE"] + FEATURE_GROUPS["WAVE_DYNAMICS"])
FEATURE_SETS["BASE_WAVE_WIND_HISTORY"] = uniq(FEATURE_GROUPS["BASE_WAVE"] + FEATURE_GROUPS["WIND_HISTORY"])
FEATURE_SETS["BASE_WAVE_PRESSURE"] = uniq(FEATURE_GROUPS["BASE_WAVE"] + FEATURE_GROUPS["PRESSURE_TENDENCY"])
FEATURE_SETS["BASE_WAVE_ALIGNMENT"] = uniq(FEATURE_GROUPS["BASE_WAVE"] + FEATURE_GROUPS["ALIGNMENT"])
FEATURE_SETS["BASE_WAVE_ATMOS"] = uniq(FEATURE_GROUPS["BASE_WAVE"] + FEATURE_GROUPS["ATMOS"])
FEATURE_SETS["WAVE_CORE"] = uniq(FEATURE_GROUPS["BASE_WAVE"] + FEATURE_GROUPS["WAVE_MOMENTUM"] + FEATURE_GROUPS["WAVE_DYNAMICS"])
FEATURE_SETS["WAVE_WIND"] = uniq(FEATURE_GROUPS["BASE_WAVE"] + FEATURE_GROUPS["WIND_BASIC"] + FEATURE_GROUPS["WAVE_DYNAMICS"])
FEATURE_SETS["WAVE_WIND_HISTORY"] = uniq(FEATURE_GROUPS["BASE_WAVE"] + FEATURE_GROUPS["WIND_BASIC"] + FEATURE_GROUPS["WIND_HISTORY"])
FEATURE_SETS["WAVE_MOMENTUM_WIND"] = uniq(FEATURE_GROUPS["BASE_WAVE"] + FEATURE_GROUPS["WAVE_MOMENTUM"] + FEATURE_GROUPS["WIND_BASIC"])
FEATURE_SETS["STORM_PHYSICS"] = uniq(FEATURE_GROUPS["BASE_WAVE"] + FEATURE_GROUPS["WIND_BASIC"] + FEATURE_GROUPS["WIND_HISTORY"] + FEATURE_GROUPS["PRESSURE_TENDENCY"] + FEATURE_GROUPS["WAVE_DYNAMICS"])
FEATURE_SETS["STORM_DIRECTION"] = uniq(FEATURE_GROUPS["BASE_WAVE"] + FEATURE_GROUPS["WIND_BASIC"] + FEATURE_GROUPS["WIND_HISTORY"] + FEATURE_GROUPS["PRESSURE_TENDENCY"] + FEATURE_GROUPS["WAVE_DYNAMICS"] + FEATURE_GROUPS["ALIGNMENT"])
FEATURE_SETS["PHYSICS_OPTIMAL_11"] = uniq(["hs", "tp", "hmax", "wspd", "u_wind", "v_wind", "hs_diff_1h", "wave_energy", "effective_wind_forcing", "u_wave", "v_wave"])
FEATURE_SETS["ALL_FEATURES"] = uniq(sum(FEATURE_GROUPS.values(), []))

for name, cols in list(FEATURE_SETS.items()):
    FEATURE_SETS[name] = [c for c in cols if c in df.columns]

In [7]:
# ============================================================
# 4. SAMPLE BUILDING & DATASET DEFINITION
# ============================================================
STATION_TO_ID = {station: i for i, station in enumerate(sorted(df[STATION_COL].unique()))}
WINDOW_META_KEYS = ("station_id", "start_idx", "end_idx", "origin_hs", "origin_time")

def build_samples(frame, features, input_len=INPUT_LEN, lead_steps=LEAD_STEPS):
    station_id, start_idx, end_idx = [], [], []
    origin_hs, origin_time = [], []
    x_by_station, hs_by_station = {}, {}
    max_lead = max(lead_steps)

    for station, g in frame.groupby(STATION_COL, sort=False):
        g = g.sort_values(TIME_COL).reset_index(drop=True)
        sid = STATION_TO_ID[station]
        Xv = g.loc[:, features].to_numpy(dtype=np.float32, copy=True)
        hsv = g["hs"].to_numpy(dtype=np.float32, copy=True)
        obs = g["hs_original_observed"].to_numpy(dtype=np.int8, copy=True)
        times = g[TIME_COL].to_numpy(copy=True)
        x_by_station[sid] = Xv
        hs_by_station[sid] = hsv
        dt = pd.Series(g[TIME_COL]).diff().dt.total_seconds().div(60).to_numpy()

        for e in range(input_len - 1, len(g) - max_lead):
            s = e - input_len + 1
            if not np.all(dt[s + 1:e + 1] == STEP_MINUTES):
                continue
            if not np.isfinite(Xv[s:e + 1]).all():
                continue
            target_idx = np.asarray([e + step for step in lead_steps], dtype=np.int64)
            if not np.isfinite(hsv[target_idx]).all() or not np.all(obs[target_idx] == 1):
                continue
            if not np.isfinite(hsv[e]):
                continue

            station_id.append(sid)
            start_idx.append(s)
            end_idx.append(e)
            origin_hs.append(hsv[e])
            origin_time.append(times[e])

    return {
        "station_id": np.asarray(station_id, dtype=np.int64),
        "start_idx": np.asarray(start_idx, dtype=np.int64),
        "end_idx": np.asarray(end_idx, dtype=np.int64),
        "origin_hs": np.asarray(origin_hs, dtype=np.float32),
        "origin_time": np.asarray(origin_time, dtype="datetime64[ns]"),
        "X_by_station": x_by_station,
        "hs_by_station": hs_by_station,
        "source_frame": frame,
        "features": list(features),
        "n_features": len(features),
    }

def chronological_split(samples, train_ratio=TRAIN_RATIO):
    times = pd.Series(pd.to_datetime(samples["origin_time"]))
    source_tz = samples["source_frame"][TIME_COL].dt.tz
    current_tz = times.dt.tz

    if source_tz is not None:
        if current_tz is None:
            times = times.dt.tz_localize(source_tz)
        else:
            times = times.dt.tz_convert(source_tz)
    elif current_tz is not None:
        times = times.dt.tz_localize(None)

    cutoff = times.quantile(train_ratio)
    train_mask = (times <= cutoff).to_numpy()
    valid_mask = (times > cutoff).to_numpy()

    def subset(mask):
        part = {key: samples[key][mask] for key in WINDOW_META_KEYS}
        part.update({
            "X_by_station": samples["X_by_station"],
            "hs_by_station": samples["hs_by_station"],
            "source_frame": samples["source_frame"],
            "features": samples["features"],
            "n_features": samples["n_features"],
            "split_cutoff": cutoff,
        })
        return part

    return subset(train_mask), subset(valid_mask), cutoff

def limit_ablation_samples(samples, max_samples=ABLATION_MAX_SAMPLES):
    n = len(samples["end_idx"])
    if n <= max_samples:
        return samples
    keep_idx = np.linspace(0, n - 1, max_samples, dtype=np.int64)
    reduced = {key: samples[key][keep_idx] for key in WINDOW_META_KEYS}
    reduced.update({
        "X_by_station": samples["X_by_station"],
        "hs_by_station": samples["hs_by_station"],
        "source_frame": samples["source_frame"],
        "features": samples["features"],
        "n_features": samples["n_features"],
    })
    return reduced

def scale_samples(train, valid):
    scaler = StandardScaler()
    train_rows = train["source_frame"][TIME_COL] <= train["split_cutoff"]
    scaler.fit(train["source_frame"].loc[train_rows, train["features"]])
    for Xv in train["X_by_station"].values():
        scaler.transform(Xv, copy=False)
    return dict(train), dict(valid), scaler

class WaveDataset(Dataset):
    def __init__(self, samples):
        self.X_by_station = samples["X_by_station"]
        self.hs_by_station = samples["hs_by_station"]
        self.station_id = samples["station_id"]
        self.start_idx = samples["start_idx"]
        self.end_idx = samples["end_idx"]
        self.origin_hs = samples["origin_hs"]
        self.lead_steps = np.asarray(LEAD_STEPS, dtype=np.int64)

    def __len__(self):
        return len(self.end_idx)

    def __getitem__(self, idx):
        sid = int(self.station_id[idx])
        start, end = int(self.start_idx[idx]), int(self.end_idx[idx])
        X = torch.from_numpy(self.X_by_station[sid][start:end + 1])
        y = torch.from_numpy(self.hs_by_station[sid][end + self.lead_steps])
        return X, y, torch.tensor(self.origin_hs[idx], dtype=torch.float32), torch.tensor(sid)

def make_loader(samples, batch_size=128, shuffle=False):
    ds = WaveDataset(samples)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=False,
        drop_last=False,
    )

In [8]:
# ============================================================
# 5. PatchTST MODEL ARCHITECTURE & LOSS
# ============================================================
class PatchTST(nn.Module):
    """
    PatchTST: A Time Series is Worth 64 Words (Channel-Independence + Patch Embedding)
    """
    def __init__(self, seq_len, n_features, pred_len, patch_len=16, stride=8, d_model=64, n_heads=4, e_layers=3, dropout=0.2, d_ff=None):
        super().__init__()
        self.seq_len = seq_len
        self.n_features = n_features
        self.pred_len = pred_len
        self.patch_len = patch_len
        self.stride = stride
        if d_ff is None:
            d_ff = d_model * 4

        # 패치 수 계산 (마지막 패치 패딩 지원)
        self.padding = stride - ((seq_len - patch_len) % stride)
        if self.padding == stride:
            self.padding = 0
        self.num_patches = (seq_len + self.padding - patch_len) // stride + 1

        # 1. 패치 임베딩: patch_len -> d_model
        self.patch_embedding = nn.Linear(patch_len, d_model)
        self.pos_embedding = nn.Parameter(torch.randn(1, self.num_patches, d_model) * 0.02)
        self.dropout = nn.Dropout(dropout)

        # 2. Transformer Encoder (Channel-Independence: 변수별로 동일 인코더 공유)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=e_layers)
        self.norm = nn.LayerNorm(d_model)

        # 3. 예측 헤드: Flatten (n_features * num_patches * d_model) -> pred_len
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_features * self.num_patches * d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, pred_len),
        )

    def forward(self, x):
        # x shape: (B, L, F)
        B, L, F = x.shape

        # 패딩 처리
        if self.padding > 0:
            pad = x[:, -1:, :].repeat(1, self.padding, 1)
            x = torch.cat([x, pad], dim=1)

        # Unfold로 패치 슬라이싱: (B, num_patches, patch_len, F)
        x = x.unfold(dimension=1, size=self.patch_len, step=self.stride)

        # Channel-Independence를 위해 차원 변환: (B * F, num_patches, patch_len)
        x = x.permute(0, 3, 1, 2).contiguous().view(B * F, self.num_patches, self.patch_len)

        # 임베딩 + 포지셔널 인코딩: (B * F, num_patches, d_model)
        x = self.patch_embedding(x) + self.pos_embedding
        x = self.dropout(x)

        # 인코더 통과
        x = self.encoder(x)
        x = self.norm(x)

        # 원래 변수 차원 복원: (B, F, num_patches, d_model)
        x = x.view(B, F, self.num_patches, -1)

        # 최종 타겟 예측 (B, pred_len)
        out = self.head(x)
        return out

class CompetitionAlignedRMSELoss(nn.Module):
    def __init__(self, threshold=1.5, high_weight=1.0, low_weight=0.3):
        super().__init__()
        self.threshold = threshold
        self.high_weight = high_weight
        self.low_weight = low_weight

    def forward(self, pred, target, origin_hs):
        diff_sq = (pred - target) ** 2
        weights = torch.where(
            origin_hs >= self.threshold,
            torch.tensor(self.high_weight, device=pred.device),
            torch.tensor(self.low_weight, device=pred.device)
        ).unsqueeze(-1)
        weighted_mse = torch.sum(weights * diff_sq) / torch.sum(weights)
        return torch.sqrt(weighted_mse + 1e-6)

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def lead_rmse(y_true, y_pred):
    return {f"rmse_{h}h": rmse(y_true[:, i], y_pred[:, i]) for i, h in enumerate(LEAD_HOURS)}

def competition_like_rmse(y_true, y_pred, origin_hs):
    mask = origin_hs >= 1.5
    if mask.sum() == 0:
        return np.nan, 0
    return rmse(y_true[mask], y_pred[mask]), int(mask.sum())

def competition_aligned_rmse(y_true, y_pred, origin_hs, threshold=1.5, high_weight=1.0, low_weight=0.3):
    diff_sq = (y_pred - y_true) ** 2
    weights = np.where(origin_hs >= threshold, high_weight, low_weight)[:, None]
    return float(np.sqrt(np.sum(weights * diff_sq) / np.sum(weights) + 1e-6))

def select_78h_separated_indices(times, station_ids, origin_hs, min_hours=78):
    selected = []
    times = pd.to_datetime(times)
    for sid in np.unique(station_ids):
        idx = np.where((station_ids == sid) & (origin_hs >= 1.5))[0]
        idx = idx[np.argsort(times[idx])]
        last_time = None
        for i in idx:
            t = times[i]
            if last_time is None or (t - last_time) >= pd.Timedelta(hours=min_hours):
                selected.append(i)
                last_time = t
    return np.asarray(selected, dtype=int)

def exact_competition_rmse(y_true, y_pred, samples):
    idx = select_78h_separated_indices(samples["origin_time"], samples["station_id"], samples["origin_hs"], min_hours=78)
    if len(idx) == 0:
        return np.nan, 0
    return rmse(y_true[idx], y_pred[idx]), len(idx)

def evaluate_model(model, loader):
    model.eval()
    preds, ys, origins = [], [], []
    with torch.no_grad():
        for X, y, origin_hs, _ in loader:
            X, y = X.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            pred = model(X)
            preds.append(pred.cpu().numpy())
            ys.append(y.cpu().numpy())
            origins.append(origin_hs.numpy())

    y_true = np.concatenate(ys)
    y_pred = np.concatenate(preds)
    origin_hs = np.concatenate(origins)

    overall = rmse(y_true, y_pred)
    comp, comp_n = competition_like_rmse(y_true, y_pred, origin_hs)
    aligned = competition_aligned_rmse(y_true, y_pred, origin_hs)
    result = {
        "overall_rmse": overall,
        "comp_rmse": comp,
        "competition_aligned_rmse": aligned,
        "comp_valid_n": comp_n,
        **lead_rmse(y_true, y_pred),
    }
    return result, y_true, y_pred

def train_one_model(train_samples, valid_samples, params, max_epochs, patience, verbose=True):
    seed_everything(SEED)
    batch_size = params.get("batch_size", 128)
    train_loader = make_loader(train_samples, batch_size=batch_size, shuffle=True)
    valid_loader = make_loader(valid_samples, batch_size=batch_size, shuffle=False)

    model = PatchTST(
        seq_len=INPUT_LEN,
        n_features=train_samples["n_features"],
        pred_len=N_TARGETS,
        patch_len=params.get("patch_len", 16),
        stride=params.get("stride", 8),
        d_model=params["d_model"],
        n_heads=params["n_heads"],
        e_layers=params["e_layers"],
        dropout=params["dropout"],
        d_ff=params.get("d_ff", params["d_model"] * 4),
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
    criterion = CompetitionAlignedRMSELoss(threshold=1.5, high_weight=1.0, low_weight=0.3)

    best_state = None
    best_score = np.inf
    wait = 0
    history = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_losses = []
        for X, y, origin_hs, _ in train_loader:
            X, y, origin_hs = X.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True), origin_hs.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            pred = model(X)
            loss = criterion(pred, y, origin_hs)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(loss.item())

        metrics, _, _ = evaluate_model(model, valid_loader)
        score = metrics["competition_aligned_rmse"]
        history.append({"epoch": epoch, "train_loss": float(np.mean(train_losses)), **metrics})

        if verbose:
            print(f"Epoch {epoch:02d} | train={np.mean(train_losses):.5f} | overall={metrics['overall_rmse']:.5f} | comp={metrics['comp_rmse']:.5f}")

        if score < best_score:
            best_score = score
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
        if wait >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    final_metrics, y_true, y_pred = evaluate_model(model, valid_loader)
    return model, final_metrics, pd.DataFrame(history), y_true, y_pred

In [9]:
# ============================================================
# 6. RAM-SAFE FEATURE ABLATION STUDY (PatchTST)
# ============================================================
FIXED_PARAMS = {
    "patch_len": 16,
    "stride": 8,
    "d_model": 64,
    "n_heads": 4,
    "e_layers": 2,
    "dropout": 0.20,
    "lr": 3e-5,
    "weight_decay": 1e-5,
    "batch_size": 64,
}

ablation_records = []

for feature_name, features in FEATURE_SETS.items():
    print("\n" + "=" * 80)
    print(f"ABLATION: {feature_name} | {len(features)} features")

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    samples = build_samples(df, features)
    original_total = len(samples["end_idx"])
    if original_total < 1000:
        print("SKIP: too few samples")
        continue

    samples = limit_ablation_samples(samples, ABLATION_MAX_SAMPLES)
    n_total = len(samples["end_idx"])

    train_s, valid_s, cutoff = chronological_split(samples, train_ratio=TRAIN_RATIO)
    train_n, valid_n = len(train_s["end_idx"]), len(valid_s["end_idx"])

    if train_n == 0 or valid_n == 0:
        print("SKIP: empty split")
        continue

    train_s, valid_s, scaler = scale_samples(train_s, valid_s)
    del samples
    gc.collect()

    model, metrics, history, y_true, y_pred = train_one_model(
        train_s, valid_s,
        params=FIXED_PARAMS,
        max_epochs=ABLATION_EPOCHS,
        patience=ABLATION_PATIENCE,
        verbose=False,
    )

    exact_comp, exact_n = exact_competition_rmse(y_true, y_pred, valid_s)
    record = {
        "feature_set": feature_name,
        "n_features": len(features),
        "original_total_n": original_total,
        "ablation_total_n": n_total,
        "train_n": train_n,
        "valid_n": valid_n,
        "cutoff": cutoff,
        **metrics,
        "exact_78h_comp_rmse": exact_comp,
        "exact_78h_n": exact_n,
    }
    ablation_records.append(record)

    del model, scaler, train_s, valid_s, history, y_true, y_pred
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

ablation_df = pd.DataFrame(ablation_records).sort_values(["competition_aligned_rmse", "overall_rmse"], ascending=[True, True]).reset_index(drop=True)
ablation_df.to_csv(EXP_DIR / "feature_ablation.csv", index=False)
display(ablation_df)


ABLATION: BASE_WAVE | 3 features

ABLATION: BASE_WAVE_WIND | 7 features

ABLATION: BASE_WAVE_MOMENTUM | 9 features

ABLATION: BASE_WAVE_DYNAMICS | 8 features

ABLATION: BASE_WAVE_WIND_HISTORY | 8 features

ABLATION: BASE_WAVE_PRESSURE | 6 features

ABLATION: BASE_WAVE_ALIGNMENT | 5 features

ABLATION: BASE_WAVE_ATMOS | 6 features

ABLATION: WAVE_CORE | 14 features

ABLATION: WAVE_WIND | 12 features

ABLATION: WAVE_WIND_HISTORY | 12 features

ABLATION: WAVE_MOMENTUM_WIND | 13 features

ABLATION: STORM_PHYSICS | 20 features

ABLATION: STORM_DIRECTION | 22 features

ABLATION: PHYSICS_OPTIMAL_11 | 11 features

ABLATION: ALL_FEATURES | 31 features


,feature_set,n_features,original_total_n,ablation_total_n,train_n,valid_n,cutoff,overall_rmse,comp_rmse,competition_aligned_rmse,comp_valid_n,rmse_3h,rmse_6h,rmse_9h,rmse_12h,rmse_18h,rmse_24h,exact_78h_comp_rmse,exact_78h_n
0,BASE_WAVE,3,129600,25000,20000,5000,2025-04-30 14:42:00+09:00,0.368625,0.521713,0.961180,328,0.257369,0.296978,0.333359,0.370495,0.444164,0.463889,0.563152,23
1,BASE_WAVE_ATMOS,6,129600,25000,20000,5000,2025-04-30 14:42:00+09:00,0.380735,0.477380,0.967372,328,0.261386,0.318615,0.351006,0.389863,0.451518,0.469946,0.607698,23
2,BASE_WAVE_PRESSURE,6,129600,25000,20000,5000,2025-04-30 14:42:00+09:00,0.387274,0.514541,0.995642,328,0.279830,0.332645,0.361029,0.389826,0.439526,0.485222,0.571639,23
3,BASE_WAVE_WIND,7,129600,25000,20000,5000,2025-04-30 14:42:00+09:00,0.388284,0.547634,1.011620,328,0.296769,0.329423,0.347975,0.380141,0.446782,0.492735,0.585857,23
4,BASE_WAVE_WIND_HISTORY,8,129600,25000,20000,5000,2025-04-30 14:42:00+09:00,0.392438,0.538373,1.015991,328,0.288354,0.313229,0.361774,0.377554,0.472834,0.495765,0.606445,23
5,BASE_WAVE_MOMENTUM,9,129600,25000,20000,5000,2025-04-30 14:42:00+09:00,0.385961,0.569565,1.016627,328,0.269972,0.329173,0.366861,0.397495,0.430843,0.484087,0.661116,23
6,WAVE_MOMENTUM_WIND,13,129600,25000,20000,5000,2025-04-30 14:42:00+09:00,0.381857,0.610413,1.027387,328,0.262597,0.304851,0.344560,0.379680,0.444532,0.502503,0.582817,23
7,WAVE_WIND_HISTORY,12,129600,25000,20000,5000,2025-04-30 14:42:00+09:00,0.390735,0.590642,1.035520,328,0.271003,0.336136,0.341939,0.393996,0.472392,0.484050,0.608507,23
8,WAVE_WIND,12,129600,25000,20000,5000,2025-04-30 14:42:00+09:00,0.388301,0.610645,1.039988,328,0.268662,0.328112,0.337582,0.380533,0.465909,0.498992,0.656761,23
9,STORM_PHYSICS,20,129600,25000,20000,5000,2025-04-30 14:42:00+09:00,0.412206,0.532031,1.053377,328,0.329991,0.338513,0.401812,0.415218,0.482311,0.479079,0.641311,23


In [10]:
# ============================================================
# 7. SELECT BEST FEATURE SET & PREPARE FULL DATA
# ============================================================
max_comp_n = ablation_df["comp_valid_n"].max()
eligible = ablation_df[ablation_df["comp_valid_n"] >= 0.80 * max_comp_n].copy()
eligible = eligible.sort_values(["competition_aligned_rmse", "overall_rmse"], ascending=[True, True])

BEST_FEATURE_NAME = eligible.iloc[0]["feature_set"]
BEST_FEATURES = FEATURE_SETS[BEST_FEATURE_NAME]

print("BEST FEATURE SET:", BEST_FEATURE_NAME)
print(f"SELECTED {len(BEST_FEATURES)} FEATURES:", BEST_FEATURES)

best_samples = build_samples(df, BEST_FEATURES)
train_samples, valid_samples, split_cutoff = chronological_split(best_samples)
train_samples, valid_samples, best_scaler = scale_samples(train_samples, valid_samples)

print(f"train samples: {len(train_samples['end_idx'])}")
print(f"valid samples: {len(valid_samples['end_idx'])}")

BEST FEATURE SET: BASE_WAVE
SELECTED 3 FEATURES: ['hs', 'tp', 'hmax']
train samples: 103680
valid samples: 25920


In [9]:
# ============================================================
# 8. OPTUNA HYPERPARAMETER TUNING (PatchTST)
# ============================================================
def objective(trial):
    patch_len = trial.suggest_categorical("patch_len", [8, 16, 24])
    stride = trial.suggest_categorical("stride", [4, 8, 12])
    d_model = trial.suggest_categorical("d_model", [64, 128, 256])
    valid_heads = [h for h in [2, 4, 8] if d_model % h == 0]
    params = {
        "patch_len": patch_len,
        "stride": stride,
        "d_model": d_model,
        "n_heads": trial.suggest_categorical("n_heads", valid_heads),
        "e_layers": trial.suggest_int("e_layers", 2, 4),
        "dropout": trial.suggest_float("dropout", 0.05, 0.35),
        "lr": trial.suggest_float("lr", 5e-6, 1e-4, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [64, 128, 256]),
    }
    try:
        model, metrics, _, _, _ = train_one_model(
            train_samples, valid_samples, params=params, max_epochs=OPTUNA_EPOCHS, patience=OPTUNA_PATIENCE, verbose=False
        )
        trial.set_user_attr("overall_rmse", metrics["overall_rmse"])
        trial.set_user_attr("competition_aligned_rmse", metrics["competition_aligned_rmse"])
        trial.set_user_attr("comp_valid_n", metrics["comp_valid_n"])
        score = metrics["competition_aligned_rmse"]
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return score
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()
            raise optuna.TrialPruned("CUDA OOM")
        raise

study = optuna.create_study(direction="minimize", sampler=TPESampler(seed=SEED), study_name="exp10_patchtst")
study.optimize(objective, n_trials=N_TRIALS, gc_after_trial=True)

trials_df = study.trials_dataframe()
trials_df.to_csv(EXP_DIR / "optuna_trials.csv", index=False)
with open(EXP_DIR / "best_params.json", "w", encoding="utf-8") as f:
    json.dump(study.best_params, f, indent=2, ensure_ascii=False)

print("BEST VALUE:", study.best_value)
print("BEST PARAMS:", study.best_params)

[I 2026-08-28 18:10:42,812] A new study created in memory with name: exp10_patchtst
[I 2026-08-28 18:22:26,880] Trial 0 finished with value: 1.0523596724760287 and parameters: {'patch_len': 16, 'stride': 4, 'd_model': 128, 'n_heads': 8, 'e_layers': 4, 'dropout': 0.11370173320348284, 'lr': 8.62044609791077e-06, 'weight_decay': 3.549878832196506e-06, 'batch_size': 128}. Best is trial 0 with value: 1.0523596724760287.
[I 2026-08-28 18:23:52,503] Trial 1 finished with value: 1.071538145949245 and parameters: {'patch_len': 16, 'stride': 12, 'd_model': 64, 'n_heads': 8, 'e_layers': 2, 'dropout': 0.06951547789558385, 'lr': 8.580222514877414e-05, 'weight_decay': 0.0007886714129990489, 'batch_size': 64}. Best is trial 0 with value: 1.0523596724760287.
[I 2026-08-28 18:28:00,839] Trial 2 finished with value: 1.1212643380728098 and parameters: {'patch_len': 8, 'stride': 12, 'd_model': 128, 'n_heads': 4, 'e_layers': 4, 'dropout': 0.28253984700833434, 'lr': 8.342309693146745e-05, 'weight_decay': 0.

BEST VALUE: 0.941708540411476
BEST PARAMS: {'patch_len': 16, 'stride': 8, 'd_model': 128, 'n_heads': 2, 'e_layers': 3, 'dropout': 0.33523709065415297, 'lr': 1.3460589138005451e-05, 'weight_decay': 4.128038469925015e-05, 'batch_size': 64}


In [11]:
# ============================================================
# 9. FINAL MODEL RETRAINING & ARTIFACTS
# ============================================================
BEST_PARAMS = {'patch_len': 16, 'stride': 8, 'd_model': 128, 'n_heads': 2, 'e_layers': 3, 'dropout': 0.33523709065415297, 'lr': 1.3460589138005451e-05, 'weight_decay': 4.128038469925015e-05, 'batch_size': 64}
final_model, final_metrics, final_history, y_true, y_pred = train_one_model(
    train_samples, valid_samples, params=BEST_PARAMS, max_epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE, verbose=True
)

exact_comp_rmse, exact_comp_n = exact_competition_rmse(y_true, y_pred, valid_samples)
final_metrics["exact_78h_comp_rmse"] = exact_comp_rmse
final_metrics["exact_78h_n"] = exact_comp_n

print("\n===== FINAL EXP10 METRICS (PatchTST) =====")
display(pd.Series(final_metrics).to_frame("value"))
final_history.to_csv(EXP_DIR / "final_history.csv", index=False)
pd.DataFrame([final_metrics]).to_csv(EXP_DIR / "final_metrics.csv", index=False)

torch.save(
    {
        "model_state_dict": final_model.state_dict(),
        "features": BEST_FEATURES,
        "feature_set_name": BEST_FEATURE_NAME,
        "input_len": INPUT_LEN,
        "lead_hours": LEAD_HOURS,
        "params": BEST_PARAMS,
        "station_to_id": STATION_TO_ID,
        "metrics": final_metrics,
    },
    EXP_DIR / "best_model.pt",
)
joblib.dump(best_scaler, EXP_DIR / "scaler.pkl")
print("Saved EXP10 artifacts to:", EXP_DIR.resolve())

Epoch 01 | train=1.78217 | overall=0.35969 | comp=0.51980
Epoch 02 | train=1.64457 | overall=0.37926 | comp=0.58471
Epoch 03 | train=1.60221 | overall=0.37279 | comp=0.56880
Epoch 04 | train=1.57281 | overall=0.41464 | comp=0.68261
Epoch 05 | train=1.54191 | overall=0.41204 | comp=0.64867
Epoch 06 | train=1.51753 | overall=0.39892 | comp=0.59308
Epoch 07 | train=1.48712 | overall=0.39895 | comp=0.56366

===== FINAL EXP10 METRICS (PatchTST) =====


,value
overall_rmse,0.359695
comp_rmse,0.519803
competition_aligned_rmse,0.941709
comp_valid_n,1672.000000
rmse_3h,0.255480
rmse_6h,0.299268
rmse_9h,0.331660
rmse_12h,0.362912
rmse_18h,0.421075
rmse_24h,0.449937


Saved EXP10 artifacts to: /content/experiments/exp10


### Save OOF Predictions

In [14]:
oof_df = pd.DataFrame({
    "origin_time": valid_samples["origin_time"],
    "station_id": valid_samples["station_id"],
    "origin_hs": valid_samples["origin_hs"],
})

# Add true and predicted values for each lead hour
for i, lead_h in enumerate(LEAD_HOURS):
    oof_df[f"y_true_{lead_h}h"] = y_true[:, i]
    oof_df[f"y_pred_{lead_h}h"] = y_pred[:, i]

oof_df.to_csv(PROJECT_ROOT / "oof" / PROJECT_ROOT / "oof" / "oof_predictions.csv", index=False)
print(f"OOF predictions saved to: {PROJECT_ROOT / "oof" / PROJECT_ROOT / "oof" / "oof_predictions.csv"}")
display(oof_df.head())

OOF predictions saved to: experiments/exp10/oof_predictions.csv


,origin_time,station_id,origin_hs,y_true_3h,y_pred_3h,y_true_6h,y_pred_6h,y_true_9h,y_pred_9h,y_true_12h,y_pred_12h,y_true_18h,y_pred_18h,y_true_24h,y_pred_24h
0,2025-04-30 15:00:00,0,1.080,1.340,1.000436,1.41,1.083133,1.460,1.247450,1.540,1.139707,0.840,1.160587,0.640,1.163210
1,2025-04-30 15:10:00,0,1.055,1.290,0.961799,1.37,1.063031,1.505,1.213298,1.515,1.086066,0.845,1.136061,0.655,1.138230
2,2025-04-30 15:20:00,0,1.030,1.240,0.930513,1.33,1.054438,1.550,1.217936,1.490,1.067721,0.850,1.119751,0.670,1.134273
3,2025-04-30 15:30:00,0,1.040,1.215,0.924528,1.34,1.065449,1.600,1.226964,1.475,1.067728,0.835,1.130498,0.735,1.149066
4,2025-04-30 15:40:00,0,1.050,1.190,0.929661,1.35,1.072744,1.650,1.218450,1.460,1.067024,0.820,1.135562,0.800,1.155348


In [13]:
# ============================================================
# 10. TEST INFERENCE & SUBMISSION CREATION (EXP10)
# ============================================================
BASE_FALLBACK_COLS = ["tp", "wspd", "gust", "wdir", "wvdir", "airt", "relh", "caph"]
station_medians = df.groupby("station")[BASE_FALLBACK_COLS].median()
global_medians = df[BASE_FALLBACK_COLS].median()

ratio_df = df[df["hs"].notna() & df["hmax"].notna() & (df["hs"] > 0)].copy()
ratio_df["hmax_hs_ratio"] = ratio_df["hmax"] / ratio_df["hs"]
ratio_df = ratio_df[ratio_df["hmax_hs_ratio"].between(1.0, 3.0)]
station_hmax_ratio = ratio_df.groupby("station")["hmax_hs_ratio"].median()
global_hmax_ratio = float(ratio_df["hmax_hs_ratio"].median())

def get_station_median(station, col):
    value = np.nan
    if station in station_medians.index and col in station_medians.columns:
        value = station_medians.loc[station, col]
    if not np.isfinite(value):
        value = global_medians[col]
    return float(value)

def get_hmax_ratio(station):
    if station in station_hmax_ratio.index:
        value = station_hmax_ratio.loc[station]
        if np.isfinite(value):
            return float(value)
    return global_hmax_ratio

def add_test_features_exp10(context):
    feature_frames = []
    for case_id, group in context.groupby("case_id", sort=False):
        g = group.sort_values("step_minute").copy()
        st = g["station"].iloc[0]

        for col in ["hs", "tp", "hmax"]:
            g.loc[g[col] <= 0, col] = np.nan
        g.loc[g["wspd"] < 0, "wspd"] = np.nan
        g.loc[g["gust"] < 0, "gust"] = np.nan
        g.loc[~g["relh"].between(0, 100), "relh"] = np.nan
        g.loc[~g["caph"].between(950, 1050), "caph"] = np.nan

        num_cols = ["hs", "tp", "hmax", "wspd", "gust", "airt", "relh", "caph"]
        g[num_cols] = g[num_cols].interpolate(method="linear", limit_direction="both").ffill().bfill()

        for col in ["wdir", "wvdir"]:
            g[col] = g[col].ffill().bfill()
            if g[col].isna().any():
                g[col] = g[col].fillna(get_station_median(st, col))

        g["wdir"] %= 360.0
        g["wvdir"] %= 360.0

        if g["hs"].isna().any():
            g["hs"] = g["hs"].ffill().bfill()

        ratio = get_hmax_ratio(st)
        hmax_missing = g["hmax"].isna() & g["hs"].notna()
        if hmax_missing.any():
            g.loc[hmax_missing, "hmax"] = g.loc[hmax_missing, "hs"] * ratio

        for col in ["tp", "wspd", "gust", "airt", "relh", "caph"]:
            if g[col].isna().any():
                g[col] = g[col].fillna(get_station_median(st, col))

        g["hs"] = g["hs"].clip(lower=0.01)
        g["tp"] = g["tp"].clip(lower=0.01)
        g["wspd"] = g["wspd"].clip(lower=0.0)
        g["gust"] = g["gust"].clip(lower=0.0)
        g["hmax"] = np.maximum(g["hmax"], g["hs"])
        g["gust"] = np.maximum(g["gust"], g["wspd"])

        wdir_rad = np.deg2rad(g["wdir"])
        wvdir_rad = np.deg2rad(g["wvdir"])
        g["u_wind"] = g["wspd"] * np.sin(wdir_rad)
        g["v_wind"] = g["wspd"] * np.cos(wdir_rad)
        g["u_wave"] = g["hs"] * np.sin(wvdir_rad)
        g["v_wave"] = g["hs"] * np.cos(wvdir_rad)

        diff = (g["wdir"] - g["wvdir"] + 180) % 360 - 180
        g["wind_wave_diff"] = np.abs(diff)
        g["wind_wave_alignment"] = np.cos(np.deg2rad(diff))

        g["hs_diff_1h"] = g["hs"] - g["hs"].shift(6)
        g["hs_diff_3h"] = g["hs"] - g["hs"].shift(18)
        g["hs_mean_6h"] = g["hs"].rolling(36, min_periods=1).mean()
        g["hs_mean_12h"] = g["hs"].rolling(72, min_periods=1).mean()
        g["hs_max_6h"] = g["hs"].rolling(36, min_periods=1).max()
        g["hs_max_12h"] = g["hs"].rolling(72, min_periods=1).max()

        g["wspd_mean_6h"] = g["wspd"].rolling(36, min_periods=1).mean()
        g["wspd_mean_12h"] = g["wspd"].rolling(72, min_periods=1).mean()
        g["gust_max_6h"] = g["gust"].rolling(36, min_periods=1).max()
        g["gust_max_12h"] = g["gust"].rolling(72, min_periods=1).max()
        g["gust_minus_wspd"] = g["gust"] - g["wspd"]

        g["caph_change_3h"] = g["caph"] - g["caph"].shift(18)
        g["caph_change_6h"] = g["caph"] - g["caph"].shift(36)
        g["caph_change_12h"] = g["caph"] - g["caph"].shift(72)

        wl = 1.56 * (g["tp"] ** 2)
        g["wave_steepness"] = g["hs"] / np.maximum(wl, 1.0)
        g["wave_energy"] = g["hs"] ** 2
        g["effective_wind_forcing"] = (g["wspd"] ** 2) * g["wind_wave_alignment"]

        g = g.replace([np.inf, -np.inf], np.nan)
        g[BEST_FEATURES] = g[BEST_FEATURES].bfill().ffill()
        feature_frames.append(g)

    return pd.concat(feature_frames, ignore_index=True)

test_context = pd.read_parquet(TEST_CONTEXT_PATH)
test_index = pd.read_csv(TEST_INDEX_PATH)
test_features = add_test_features_exp10(test_context)

case_order = test_index["case_id"].drop_duplicates().tolist()
windows = []
for case_id in case_order:
    grp = test_features[test_features["case_id"] == case_id].sort_values("step_minute")
    X = grp[BEST_FEATURES].to_numpy(dtype=np.float32)
    windows.append(X)

X_test_raw = np.stack(windows)
X_test = best_scaler.transform(X_test_raw.reshape(-1, len(BEST_FEATURES))).reshape(X_test_raw.shape).astype(np.float32)

final_model.eval()
pred_batches = []
INFER_BATCH_SIZE = 128
with torch.no_grad():
    for start in range(0, len(X_test), INFER_BATCH_SIZE):
        xb = torch.from_numpy(X_test[start:start + INFER_BATCH_SIZE]).to(DEVICE)
        pred = final_model(xb).detach().cpu().numpy()
        pred_batches.append(pred)

preds = np.concatenate(pred_batches, axis=0)

pred_rows = []
for case_id, row in zip(case_order, preds):
    for lead_h, val in zip(LEAD_HOURS, row):
        pred_rows.append({"case_id": case_id, "lead_h": lead_h, "hs_pred": float(val)})

submission = test_index.merge(pd.DataFrame(pred_rows), on=["case_id", "lead_h"], how="left", validate="one_to_one")
submission["hs_pred"] = submission["hs_pred"].clip(lower=0.0, upper=30.0)
submission.to_csv(SUBMISSION_PATH, index=False, encoding="utf-8")

print("\n" + "=" * 80)
print(f"EXP10 SUBMISSION SAVED: {SUBMISSION_PATH}")
print("=" * 80)
display(submission.head(12))


EXP10 SUBMISSION SAVED: submission_exp10.csv


,case_id,station,lead_h,hs_pred
0,C0001,S-ORS,3,1.978981
1,C0001,S-ORS,6,1.953448
2,C0001,S-ORS,9,1.951894
3,C0001,S-ORS,12,1.923090
4,C0001,S-ORS,18,1.674828
5,C0001,S-ORS,24,1.540651
6,C0002,I-ORS,3,1.555689
7,C0002,I-ORS,6,1.596715
8,C0002,I-ORS,9,1.752817
9,C0002,I-ORS,12,1.597206
